In [1]:
from typing import TypedDict
import requests
import os
from langgraph.graph import StateGraph, END 

In [2]:
SAIS_APP_URL = os.getenv("SAIS_APP_URL")
SAIS_TOKEN = os.getenv("SAIS_TOKEN")
SAIS_MODEL = os.getenv("SAIS_MODEL", "gpt-4.1")

if not SAIS_APP_URL or not SAIS_TOKEN:
    raise SystemExit("Error: SAIS_APP_URL and SAIS_TOKEN environment variables must be set.")

PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")
PROXY_ENABLED = os.getenv("PROXY_ENABLED", "false").lower() == "true"
SSL_VERIFY = os.getenv("SSL_VERIFY", "true").lower() == "true"

In [3]:
def generate_answer(context: str, query: str) -> str:
    response = requests.post(
        f"{SAIS_APP_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {SAIS_TOKEN}",
            "Content-Type": "application/json",
            "ApplicationType": "BRProduct",
        },
        json={
            "model": SAIS_MODEL,
            "instructions": "You are an AI technical support assistant.\n\nAnswer the user's question to the best of your knowledge.",
            "input": context + "\n\n" + query,
        },
        verify=SSL_VERIFY,
        proxies={
            "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
        }
    )
    if response.status_code == 200:
        response_data = response.json()
        try:
            outputs = response_data["body"]["output"]
            texts = []
            for item in outputs:
                if item.get("type") == "message":
                    for content in item.get("content", []):
                        if content.get("type") == "output_text":
                            texts.append(content.get("text", ""))
            if texts:
                return "\n".join(texts)
            return "Error: No output text found in the response."
        except KeyError:
            return "Error: Unexpected response format."
    else:
        return f"Error: Request failed with status code {response.status_code}"

In [4]:
class Mystate(TypedDict):
    topic: str
    response: str

def generate_llm_outline(state:Mystate) -> Mystate:
    context = "You are an AI technical support assistant. Generate a concise outline for the topic provided."
    response = generate_answer(context, state["topic"])
    return Mystate(topic=state["topic"], response=response)

In [5]:
def generate_blog(state:Mystate) -> Mystate:
    context = "You are an AI technical support assistant. Generate a detailed blog post for the topic outline provided."
    response = generate_answer(context, state["response"])
    return Mystate(topic=state["topic"], response=response)

graph = StateGraph(Mystate)
graph.add_node("generate_llm_outline", generate_llm_outline)
graph.add_node("generate_blog", generate_blog)
graph.set_entry_point("generate_llm_outline")
graph.add_edge("generate_llm_outline", "generate_blog")
graph.add_edge("generate_blog", END)

app = graph.compile()

initial_state = {
    "topic":"The topic of the blog post is 'The Future of AI in Healthcare'.",
    "response":""   
}

app_result = app.invoke(initial_state)
print("Final Result:", app_result)  

c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Final Result: {'topic': "The topic of the blog post is 'The Future of AI in Healthcare'.", 'response': '# The Future of AI in Healthcare: Opportunities, Challenges, and What Comes Next\n\nArtificial intelligence is rapidly transforming healthcare. From helping doctors detect disease earlier to streamlining hospital operations, AI is becoming an increasingly important part of modern medicine. While it is not a replacement for healthcare professionals, AI has the potential to support better decision-making, improve patient outcomes, and make care more efficient and accessible.\n\nThe future of AI in healthcare matters because it affects everyone involved in the healthcare ecosystem: patients seeking timely and accurate care, clinicians managing complex workloads, hospitals trying to improve efficiency, and healthcare systems looking for ways to reduce costs while improving quality. As AI tools become more advanced, the key question is not whether AI will influence healthcare, but how it 